¿Podemos explicar la precisión final de un prediction market a partir de la forma en que evoluciona su probabilidad?

In [5]:
import pandas as pd
import numpy as np

prices = pd.read_csv("../data/raw/prices_raw.csv")
market_dataset = pd.read_csv("../data/processed/market_dataset.csv")
df_model = pd.read_csv('../data/processed/df_model.csv')
prices["timestamp"] = pd.to_datetime(prices["timestamp"])
prices = prices.sort_values(["market_id","timestamp"])
prices["prob_return"] = (prices.groupby("market_id")["implied_probability"].diff())

In [6]:
prices

,timestamp,implied_probability,market_id,question,slug,prob_return
0,2023-02-07 01:00:32+00:00,0.500,248594,Will Hunter Biden be federally indicted by May...,will-hunter-biden-be-federally-indicted-by-may...,NaN
1,2023-02-07 02:00:53+00:00,0.500,248594,Will Hunter Biden be federally indicted by May...,will-hunter-biden-be-federally-indicted-by-may...,0.000
2,2023-02-07 03:00:34+00:00,0.500,248594,Will Hunter Biden be federally indicted by May...,will-hunter-biden-be-federally-indicted-by-may...,0.000
3,2023-02-07 04:00:05+00:00,0.500,248594,Will Hunter Biden be federally indicted by May...,will-hunter-biden-be-federally-indicted-by-may...,0.000
4,2023-02-07 05:00:07+00:00,0.500,248594,Will Hunter Biden be federally indicted by May...,will-hunter-biden-be-federally-indicted-by-may...,0.000
...,...,...,...,...,...,...
100637,2024-03-11 19:00:02+00:00,0.555,255410,US inflation >0.4% from Feb to March 2024?,us-inflation-0pt4-from-feb-to-march-2024,-0.005
100638,2024-03-11 20:00:02+00:00,0.540,255410,US inflation >0.4% from Feb to March 2024?,us-inflation-0pt4-from-feb-to-march-2024,-0.015
100639,2024-03-11 21:00:02+00:00,0.530,255410,US inflation >0.4% from Feb to March 2024?,us-inflation-0pt4-from-feb-to-march-2024,-0.010
100640,2024-03-11 22:00:02+00:00,0.530,255410,US inflation >0.4% from Feb to March 2024?,us-inflation-0pt4-from-feb-to-march-2024,0.000


In [7]:
# entropy 
from scipy.stats import entropy

def compute_entropy(returns):
    returns = returns.dropna()
    if len(returns) < 10:
        return np.nan
    hist, _ = np.histogram(returns,bins=10)
    probs = hist / hist.sum()
    probs = probs[probs > 0]
    return entropy(probs)

entropy_feature = (prices.groupby("market_id")["prob_return"].apply(compute_entropy).
rename("shannon_entropy"))

In [8]:
# Skewness
from scipy.stats import skew
skewness_feature = (prices.groupby("market_id")["prob_return"].apply(lambda x: skew(x.dropna())).rename("skewness")
)

In [9]:
# Kurtosis
from scipy.stats import kurtosis
kurtosis_feature = (prices.groupby("market_id")["prob_return"].apply(lambda x: kurtosis(x.dropna()))
    .rename("kurtosis"))

In [10]:
# Autocorrelation
def lag1_autocorr(series):
    series = series.dropna()
    if len(series) < 10:
        return np.nan
    return series.autocorr(lag=1)

autocorr_feature = (prices.groupby("market_id")["prob_return"].apply(lag1_autocorr).rename("autocorrelation")
)

In [11]:
# new features
new_features = pd.concat([entropy_feature,skewness_feature,kurtosis_feature,autocorr_feature],axis=1
).reset_index()

In [12]:
new_features.head()

,market_id,shannon_entropy,skewness,kurtosis,autocorrelation
0,248594,0.094783,-8.209442,571.526524,-0.001674
1,249778,0.367649,-0.070942,7.681394,-0.584862
2,250474,0.367649,-1.194920,8.169362,-0.008684
3,251025,0.418053,-3.037167,48.948549,0.156657
4,251027,0.573083,2.116902,16.925633,-0.257660


In [13]:
df_model_v2 = df_model.merge(new_features,on="market_id",how="left")

In [14]:
features = [
    "realized_volatility",
    "probability_range",
    "trend",
    "max_drawdown",
    "reversals",
    "shannon_entropy",
    "skewness",
    "kurtosis",
    "autocorrelation"
]
(df_model_v2[features + ["abs_surprise"]].corr()["abs_surprise"].sort_values())

max_drawdown          -0.755926
probability_range     -0.587153
reversals             -0.370240
kurtosis              -0.305769
realized_volatility   -0.143927
skewness              -0.125336
autocorrelation       -0.095746
trend                 -0.078924
shannon_entropy        0.233897
abs_surprise           1.000000
Name: abs_surprise, dtype: float64

In [15]:
high_drawdown = (df_model_v2["max_drawdown"]> df_model_v2["max_drawdown"].median())

In [18]:
df_model['abs_surprise']

0     0.0200
1     0.4500
2     0.3600
3     0.0050
4     0.1750
5     0.0600
6     0.4450
7     0.7650
8     0.0100
9     0.0060
10    0.0050
11    0.0150
12    0.0460
13    0.0155
14    0.0070
15    0.0075
16    0.0020
17    0.0075
18    0.4350
19    0.2350
20    0.2250
21    0.0005
22    0.0005
23    0.0650
24    0.5450
25    0.0005
26    0.0070
27    0.0055
28    0.0035
29    0.0005
30    0.0005
31    0.0005
32    0.0005
33    0.0005
34    0.0005
35    0.0005
36    0.5000
37    0.0075
38    0.0500
39    0.0135
40    0.0005
41    0.0005
42    0.5350
Name: abs_surprise, dtype: float64

In [19]:
from scipy.stats import mannwhitneyu
median_dd = df_model_v2["max_drawdown"].median()
high_dd = df_model_v2.loc[df_model_v2["max_drawdown"] > median_dd,"abs_surprise"]

low_dd = df_model_v2.loc[df_model_v2["max_drawdown"] <= median_dd,"abs_surprise" ]

mannwhitneyu(high_dd,low_dd,alternative="two-sided")

MannwhitneyuResult(statistic=np.float64(127.5), pvalue=np.float64(0.01201334413167982))

In [21]:
high_dd.mean()


np.float64(0.018785714285714284)

In [22]:
low_dd.mean()


np.float64(0.21090909090909085)

In [ ]:
high_dd.median()


np.float64(0.005)

In [24]:
low_dd.median()

np.float64(0.0575)

Markets that aggressively revise beliefs in response to new information converge to substantially more accurate forecasts than markets that remain anchored to their initial beliefs.

In [27]:
df_model_v2.to_csv('../data/processed/df_model_v2.csv', index = False)